# Document Navigator — Demo Notebook

**Document Navigator** is a transparent, locally-run RAG assistant over a PDF corpus. It retrieves relevant chunks from a FAISS vector index built from `documents/`, evaluates evidence strength, and generates grounded answers that cite their sources in `[filename.pdf:page]` format. Queries without sufficient evidence are refused before the LLM is ever called, eliminating hallucination on out-of-scope questions. The full stack runs offline: FAISS + MiniLM-L6-v2 for retrieval, Ollama (qwen2.5:7b) for generation — no API keys required.

This notebook walks through the pipeline in linear order: ingestion configuration → retrieval with similarity traces → grounded generation for a strong-evidence case and a refusal case → evaluation results from the 20-row eval harness. All pipeline logic lives in `src/`; this notebook only imports and narrates it. If the notebook and the code drift, the notebook is wrong.

In [1]:
import os, sys, json
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)
print(f'Working directory: {Path.cwd()}')

from src.ingest import DEFAULT_CHUNK_SIZE, DEFAULT_CHUNK_OVERLAP, DEFAULT_EMBEDDING_MODEL
from src.retrieve import Retriever, run_query
from src.generate import generate_answer
import pandas as pd

Working directory: /Users/harshith/shadow/document-navigator


## Section 1: Ingestion

`src/ingest.py` loads every PDF in `documents/` page-by-page using `PyPDFLoader`, splits them into overlapping character-level chunks with `RecursiveCharacterTextSplitter`, embeds each chunk with sentence-transformers/all-MiniLM-L6-v2 (unit-normalized so inner product equals cosine similarity), and persists the resulting FAISS index to `db_faiss/`. Each chunk carries `source`, `page`, `chunk_id`, and `chunk_index` metadata used for citation and tracing.

Ingestion is a one-time setup step — the index is already built and the cells below read from it. To rebuild from scratch: `python -m src.ingest`.

In [2]:
print(f'Chunk size   : {DEFAULT_CHUNK_SIZE} characters')
print(f'Chunk overlap: {DEFAULT_CHUNK_OVERLAP} characters')
print(f'Embedding    : {DEFAULT_EMBEDDING_MODEL}')
print()
print('Index built via: python -m src.ingest')
print('Index location : db_faiss/index.faiss + db_faiss/index.pkl')

Chunk size   : 800 characters
Chunk overlap: 120 characters
Embedding    : sentence-transformers/all-MiniLM-L6-v2

Index built via: python -m src.ingest
Index location : db_faiss/index.faiss + db_faiss/index.pkl


## Section 2: Retrieval with Traces

`src/retrieve.py` exposes a `Retriever` class that lazy-loads the FAISS index and embedding model on first use. Similarity scores are cosine similarity derived from L2-normalized embeddings: for unit vectors, `‖a − b‖² = 2 − 2·cos(a, b)`, so `cos = 1 − distance/2`, clamped to [0, 1]. Each `run_query` call returns a `RetrievalTrace` dataclass containing ranked `RetrievalHit` objects with similarity scores, source metadata, and the full chunk text. Traces can optionally be appended to `logs/retrieval_traces.jsonl`; `write_jsonl_trace=False` is used throughout this notebook to avoid polluting production logs.

In [3]:
retriever = Retriever()
retriever._ensure_loaded()  # force eager load; lazy by design, explicit here

trace = run_query(
    'How long does standard shipping take?',
    k=5,
    retriever=retriever,
    write_jsonl_trace=False,
)
print(f'Query   : {trace.query!r}')
print(f'Hits    : {len(trace.hits)}')
print(f'Elapsed : {trace.elapsed_ms:.1f} ms')

2026-05-20 01:05:51,335 | INFO    | Loading embedding model: sentence-transformers/all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

2026-05-20 01:06:00,047 | INFO    | Loading FAISS index from: /Users/harshith/shadow/document-navigator/db_faiss


2026-05-20 01:06:00,094 | INFO    | Index loaded successfully.


2026-05-20 01:06:00,106 | INFO    | Query 'How long does standard shipping take?' → 5 hit(s) in 11.6 ms


Query   : 'How long does standard shipping take?'
Hits    : 5
Elapsed : 11.6 ms


In [4]:
rows = []
for hit in trace.hits:
    preview = hit.text[:100] + ('\u2026' if len(hit.text) > 100 else '')
    rows.append({
        'rank': hit.rank,
        'source': hit.source,
        'page': hit.page,
        'similarity': round(hit.similarity, 3),
        'chunk_id': hit.chunk_id,
        'text_preview': preview,
    })
df = pd.DataFrame(rows)
df

,rank,source,page,similarity,chunk_id,text_preview
0,1,policy_shipping_returns.pdf,1,0.733,0,Shipping & Returns Policy (Sample)\n1. Standar...
1,2,policy_payments_security.pdf,1,0.263,0,Payments & Security Policy (Sample)\n1. Accept...
2,3,guide_chunking_strategy.pdf,1,0.091,0,Chunking Strategy Notes (Sample)\n1. Typical c...
3,4,policy_privacy_data_use.pdf,1,0.083,0,Privacy & Data Use Policy (Sample)\n1. Collect...
4,5,guide_logging_monitoring.pdf,1,0.012,0,Logging & Monitoring for LLM Apps (Sample)\n1....


## Section 3: Grounded Generation

`src/generate.py` gates on evidence strength before calling the LLM. The top-1 similarity score determines the path: **≥ 0.40** → `strong` evidence (proceed); **0.25 – 0.39** → `weak` evidence (proceed, flag uncertainty); **< 0.25** → `none` — refuse immediately, return the refusal message, and **skip the LLM call entirely**. This pre-LLM refusal eliminates any chance the model answers from training data when the index holds no relevant evidence. Strict mode raises the thresholds to 0.40 / 0.55.

In [5]:
result = generate_answer(
    'How long does standard shipping take?',
    retriever=retriever,
    write_jsonl_trace=False,
)
print(f'Answer           : {result.answer}')
print(f'Citations        : {result.citations_used}')
print(f'Evidence strength: {result.evidence_strength}')
print(f'Top similarity   : {round(result.top_similarity, 3)}')

2026-05-20 01:06:00,126 | INFO    | Query 'How long does standard shipping take?' → 5 hit(s) in 3.3 ms


2026-05-20 01:06:00,127 | INFO    | Calling qwen2.5:7b (temp=0.0, evidence=strong, top_sim=0.7329)


2026-05-20 01:06:08,074 | INFO    | Generated answer in 7927.0 ms; citations: ['[policy_shipping_returns.pdf:1]']


Answer           : Standard delivery takes 3–6 business days depending on location. [policy_shipping_returns.pdf:1]
Citations        : ['[policy_shipping_returns.pdf:1]']
Evidence strength: strong
Top similarity   : 0.733


In [6]:
result_refused = generate_answer(
    'What is the capital of France?',
    retriever=retriever,
    write_jsonl_trace=False,
)
print(f'Answer           : {result_refused.answer}')
print(f'Citations        : {result_refused.citations_used}')
print(f'Evidence strength: {result_refused.evidence_strength}')
print(f'Top similarity   : {round(result_refused.top_similarity, 3)}')
print(f'LLM was called   : {not result_refused.refused}')

2026-05-20 01:06:08,119 | INFO    | Query 'What is the capital of France?' → 5 hit(s) in 24.2 ms


2026-05-20 01:06:08,120 | INFO    | Refusing query 'What is the capital of France?': top_sim=0.0414 < min_sim=0.25


Answer           : I don't have enough information in the indexed documents to answer this question.
Citations        : []
Evidence strength: none
Top similarity   : 0.041
LLM was called   : False


## Section 4: Evaluation

`eval/evaluate.py` replays every row in `eval/eval_set.csv` through the full pipeline and scores retrieval, generation, and safety. The 20-row eval set covers 15 answer questions and 5 adversarial rows (3 out-of-scope, 2 prompt-injection attempts). Headline results: **precision@3 = 1.00**, **precision@5 = 1.00** (the correct source appeared in the top 5 for every answer question), **answer pass rate = 14/15** after correcting for rubric artifacts (paraphrase mismatches and multi-valued gold answers that were scored incorrectly by the single-gold-phrase rubric), and **refusal correctness = 5/5** across all 5 adversarial rows — every adversarial query landed below the 0.25 similarity floor and none reached the LLM. See `reports/retrieval_report.md` for the full findings, failure analysis, limitations, and next steps.

In [7]:
with open('reports/eval_summary.json', encoding='utf-8') as f:
    summary = json.load(f)
print(json.dumps(summary, indent=2))

print()
df_eval = pd.read_csv('reports/eval_results.csv')
df_eval.head(10)

{
  "n_rows": 20,
  "n_answer_rows": 15,
  "n_refuse_rows": 5,
  "precision_at_1": 0.8667,
  "precision_at_3": 1.0,
  "precision_at_5": 1.0,
  "citation_accuracy": 0.8,
  "key_phrase_accuracy": 0.7333,
  "answer_pass_rate": 0.7333,
  "refusal_correctness": 1.0,
  "false_refusal_rate": 0.0,
  "overall_pass_rate": 0.8,
  "refusal_rate": 0.25,
  "mean_top_similarity": 0.4247,
  "mean_elapsed_ms_total": 3462.07,
  "evidence_strength_counts": {
    "strong": 12,
    "weak": 3,
    "none": 5
  },
  "evidence_strength_by_behavior": {
    "answer": {
      "strong": 12,
      "weak": 3,
      "none": 0
    },
    "refuse": {
      "strong": 0,
      "weak": 0,
      "none": 5
    }
  }
}



,id,question,gold_citation,gold_key_phrase,expected_behavior,gold_filename,retrieved_top_k,generated_answer,citations_used,evidence_strength,top_similarity,refused,gold_in_top_1,gold_in_top_3,gold_in_top_5,gold_citation_in_answer,key_phrase_in_answer,passed,elapsed_ms_total
0,Q01,What is the standard delivery timeline?,[policy_shipping_returns.pdf:1],Standard delivery takes 3–6 business days,answer,policy_shipping_returns.pdf,rank=1 source=policy_shipping_returns.pdf page...,Standard delivery takes 3–6 business days depe...,[policy_shipping_returns.pdf:1],strong,0.4201,False,True,True,True,True,True,True,15777.24
1,Q02,What is the return window for most products?,[policy_shipping_returns.pdf:1],returned within 7 days,answer,policy_shipping_returns.pdf,rank=1 source=policy_shipping_returns.pdf page...,Most products can be returned within 7 days if...,[policy_shipping_returns.pdf:1],weak,0.3350,False,True,True,True,True,True,True,3671.39
2,Q03,How long do refunds typically take after quali...,[policy_shipping_returns.pdf:1],3–7 business days after quality check,answer,policy_shipping_returns.pdf,rank=1 source=policy_shipping_returns.pdf page...,Refunds are processed within 3–7 business days...,[policy_shipping_returns.pdf:1],strong,0.5987,False,True,True,True,True,True,True,3802.10
3,Q04,Name two accepted payment methods.,[policy_payments_security.pdf:1],"cards, UPI",answer,policy_payments_security.pdf,rank=1 source=policy_payments_security.pdf pag...,Two accepted payment methods are cards and UPI...,[policy_payments_security.pdf:1],strong,0.5595,False,True,True,True,True,False,False,3382.66
4,Q05,When is Cash on Delivery available?,[policy_payments_security.pdf:1],eligible pin codes and products,answer,policy_payments_security.pdf,rank=1 source=policy_payments_security.pdf pag...,Cash on Delivery may be available for eligible...,[policy_payments_security.pdf:1],strong,0.4075,False,True,True,True,True,True,True,3570.12
5,Q06,What is one benefit of citations in a RAG assi...,[guide_rag_basics.pdf:1],enable verification,answer,guide_rag_basics.pdf,rank=1 source=guide_rag_basics.pdf page=1 sim=...,Citations improve trust and enable verificatio...,[guide_rag_basics.pdf:1],strong,0.6391,False,True,True,True,True,True,True,3526.96
6,Q07,What does Precision@k measure?,[guide_evaluation_metrics.pdf:1],how many of the top-k retrieved chunks are rel...,answer,guide_evaluation_metrics.pdf,rank=1 source=guide_evaluation_metrics.pdf pag...,Precision@k measures how many of the top-k ret...,[guide_evaluation_metrics.pdf:1],strong,0.4705,False,True,True,True,True,True,True,3662.29
7,Q08,What chunk size range is recommended for narra...,[guide_chunking_strategy.pdf:1],500 to 800 tokens,answer,guide_chunking_strategy.pdf,rank=1 source=guide_chunking_strategy.pdf page...,Typical chunk sizes for narrative PDFs range f...,[guide_chunking_strategy.pdf:1],strong,0.8819,False,True,True,True,True,True,True,4067.67
8,Q09,Why use chunk overlap?,[guide_chunking_strategy.pdf:1],preserve context across chunk boundaries,answer,guide_chunking_strategy.pdf,rank=1 source=guide_chunking_strategy.pdf page...,Chunk overlap can improve recall by ensuring t...,[guide_rag_basics.pdf:1],strong,0.4486,False,True,True,True,False,False,False,4289.25
9,Q10,What is hybrid retrieval?,[guide_vector_search.pdf:1],merges BM25 and vector search,answer,guide_vector_search.pdf,rank=1 source=guide_evaluation_metrics.pdf pag...,Hybrid retrieval merges BM25 and vector search...,[guide_vector_search.pdf:1],weak,0.3838,False,False,True,True,True,True,True,3831.51


## Section 5: Upload mode (demo-only)

The Streamlit app supports an optional upload mode where a user supplies their own PDFs and queries a session-scoped in-memory index. The upload pipeline reuses the same chunking, embedding, retrieval, and generation logic as the persistent corpus — only the source changes. Programmatically, the entry point is `src.upload_index.build_index_from_uploads`, which accepts a list of `(filename, bytes)` tuples and returns an in-memory FAISS index plus per-file summaries. Wrap that index with `Retriever.from_vectorstore(...)` and pass it to `generate_answer` to query it. The notebook does not execute this path because it requires uploaded files; the Streamlit app at `app.py` is where the feature is exercised interactively.

## Closing

The full picture lives in three places:

- **`reports/retrieval_report.md`** — complete findings, failure analysis, limitations, and next steps including the Q09 distractor-confusion fix, paraphrase-tolerant scoring expansion, and the plan for higher-similarity injection cases that would exercise the LLM-layer prompt-injection defenses.
- **`app.py`** — interactive Streamlit UI (`streamlit run app.py`) with sidebar controls for top-k, model, temperature, and strict-evidence mode, plus per-query evidence-strength badges and per-chunk expanders. The app supports two modes — querying the evaluated persistent corpus or uploading your own PDFs for a session-scoped index.
- **`src/`** — the pipeline implementation: `ingest.py`, `retrieve.py`, `generate.py`. All notebook cells import from here; nothing in this notebook duplicates pipeline logic.